# 23 — End-to-End PyTorch Project: From Data to Reliable Model

In the previous notebooks, we learned the pieces of a PyTorch workflow separately:

- Tensors
- Autograd
- Neural networks
- Loss functions
- Optimizers
- DataLoaders
- Training loops
- CNNs
- Regularization
- Debugging
- Transfer learning
- Evaluation metrics
- Explainability

Now we will connect those pieces into a complete project.

This notebook is intentionally designed like a small real-world machine-learning study.

## In this notebook, we will study:

1. Defining the prediction task
2. Designing train / validation / test splits
3. Building a reusable dataset pipeline
4. Creating a baseline model
5. Training and validation
6. Choosing metrics
7. Checkpointing
8. Hyperparameter experiments
9. Transfer-learning baseline
10. Error analysis
11. Explainability checks
12. Reproducibility
13. Saving predictions and experiment metadata
14. Final test evaluation
15. Organizing a complete PyTorch project
16. Preparing the workflow for a real ultrasound classification project

## Main Goal

By the end of this notebook, you should understand the complete lifecycle:

$$
\boxed{
\text{Question}
\rightarrow
\text{Data}
\rightarrow
\text{Split}
\rightarrow
\text{Pipeline}
\rightarrow
\text{Baseline}
\rightarrow
\text{Training}
\rightarrow
\text{Validation}
\rightarrow
\text{Model Selection}
\rightarrow
\text{Final Test}
}
$$

The central principle is:

> **A reliable model is the result of a reliable experimental design.**


In [ ]:
import copy
import json
import math
import random
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

print("PyTorch:", torch.__version__)


# 1. Start With the Prediction Task

Before choosing a CNN, define the actual question.

For example:

> Given one ultrasound image, predict whether the case belongs to class 0, 1, or 2.

Important task-definition questions include:

- What is the input?
- What is the target?
- What is the unit of prediction?
- Is the task binary or multi-class?
- Is the output per image, study, or patient?
- What will count as success?


# 2. Define the Unit of Analysis

This is especially important in medical imaging.

Possible units include:

$$
\begin{array}{|c|c|}
\hline
\textbf{Unit} & \textbf{Meaning} \\
\hline
Image & One frame / image \\
\hline
Study & One imaging examination \\
\hline
Patient & One person \\
\hline
\end{array}
$$

If your clinical decision is per patient, a per-image model may need an aggregation rule later.


# 3. Define the Target Clearly

Suppose our example has three classes:

$$
\begin{array}{|c|c|}
\hline
0 & \text{Class A} \\
\hline
1 & \text{Class B} \\
\hline
2 & \text{Class C} \\
\hline
\end{array}
$$

The final layer must produce:

$$
\boxed{
3\ logits
}
$$

per sample.


# 4. Define the Primary Metric Before Training

A strong project decides the primary metric before final test evaluation.

Examples:

- Accuracy
- Macro F1
- Sensitivity
- Specificity
- AUROC
- AUPRC

For this synthetic multi-class example, we will use:

> **Validation loss for checkpointing**

and monitor:

> **Accuracy and macro F1**

as evaluation metrics.


# 5. Why Split Design Comes Before Modeling

If the split is wrong, a powerful model only learns a flawed experiment.

For medical imaging, leakage can happen when:

- Same patient appears in train and validation
- Neighboring frames are split separately
- Same study appears in multiple splits
- Augmented copies cross splits
- Site/device correlates with label


# 6. Synthetic Patient-Level Dataset

We will create a small synthetic image dataset.

Each patient contributes multiple images.

That lets us demonstrate:

> **Patient-level splitting**

rather than random image-level splitting.


In [ ]:
def make_base_pattern(
    class_index,
    image_size=64
):
    image = torch.zeros(
        1,
        image_size,
        image_size
    )

    center = image_size // 2

    if class_index == 0:
        image[
            :,
            10:image_size - 10,
            center - 3:center + 3
        ] = 1.0

    elif class_index == 1:
        image[
            :,
            center - 3:center + 3,
            10:image_size - 10
        ] = 1.0

    elif class_index == 2:
        image[
            :,
            center - 9:center + 9,
            center - 9:center + 9
        ] = 1.0

    else:
        raise ValueError(
            "class_index must be 0, 1, or 2"
        )

    return image


# 7. Add Patient-Specific Variation

Each patient will have:

- Small translation
- Intensity variation
- Noise
- Multiple correlated images

This is closer to real imaging data than independent random samples.


In [ ]:
def make_patient_images(
    patient_id,
    class_index,
    images_per_patient=4,
    image_size=64
):
    generator = torch.Generator()
    generator.manual_seed(
        1000 + patient_id
    )

    base = make_base_pattern(
        class_index,
        image_size=image_size
    )

    images = []

    for image_index in range(
        images_per_patient
    ):
        shift_y = int(
            torch.randint(
                -5,
                6,
                (1,),
                generator=generator
            ).item()
        )

        shift_x = int(
            torch.randint(
                -5,
                6,
                (1,),
                generator=generator
            ).item()
        )

        image = torch.roll(
            base,
            shifts=(
                shift_y,
                shift_x
            ),
            dims=(
                1,
                2
            )
        )

        gain = (
            0.85
            + 0.3
            * torch.rand(
                1,
                generator=generator
            ).item()
        )

        noise = torch.randn(
            image.shape,
            generator=generator
        ) * 0.12

        image = (
            image
            * gain
            + noise
        ).clamp(
            0.0,
            1.0
        )

        images.append(
            image
        )

    return images


# 8. Create Patient Records


In [ ]:
random.seed(42)

num_patients = 90
images_per_patient = 4

records = []

for patient_id in range(
    num_patients
):
    class_index = (
        patient_id
        % 3
    )

    patient_images = (
        make_patient_images(
            patient_id,
            class_index,
            images_per_patient=(
                images_per_patient
            )
        )
    )

    for image_index, image in enumerate(
        patient_images
    ):
        records.append({
            "patient_id":
                patient_id,

            "image_index":
                image_index,

            "label":
                class_index,

            "image":
                image
        })

print(
    "Total images:",
    len(records)
)


# 9. Visualize Example Images


In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

for class_index in range(3):
    example = next(
        record
        for record in records
        if record["label"] == class_index
    )

    axes[
        class_index
    ].imshow(
        example[
            "image"
        ].squeeze(0).numpy(),
        cmap="gray"
    )

    axes[
        class_index
    ].set_title(
        f"Class {class_index}"
    )

    axes[
        class_index
    ].axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 10. Patient-Level Split

We split patients first.

Then all images from one patient remain in the same split.

We will use approximately:

$$
70\%
$$

training,

$$
15\%
$$

validation,

and:

$$
15\%
$$

test.


In [ ]:
all_patient_ids = list(
    range(num_patients)
)

random.Random(
    42
).shuffle(
    all_patient_ids
)

train_end = int(
    0.70
    * num_patients
)

val_end = int(
    0.85
    * num_patients
)

train_patient_ids = set(
    all_patient_ids[
        :train_end
    ]
)

val_patient_ids = set(
    all_patient_ids[
        train_end:val_end
    ]
)

test_patient_ids = set(
    all_patient_ids[
        val_end:
    ]
)

print(
    len(train_patient_ids),
    len(val_patient_ids),
    len(test_patient_ids)
)


# 11. Verify No Patient Leakage


In [ ]:
assert train_patient_ids.isdisjoint(
    val_patient_ids
)

assert train_patient_ids.isdisjoint(
    test_patient_ids
)

assert val_patient_ids.isdisjoint(
    test_patient_ids
)

print(
    "Patient-level split verified."
)


# 12. Convert Records Into Split Lists


In [ ]:
train_records = [
    record
    for record in records
    if record[
        "patient_id"
    ] in train_patient_ids
]

val_records = [
    record
    for record in records
    if record[
        "patient_id"
    ] in val_patient_ids
]

test_records = [
    record
    for record in records
    if record[
        "patient_id"
    ] in test_patient_ids
]

print(
    "Train images:",
    len(train_records)
)

print(
    "Validation images:",
    len(val_records)
)

print(
    "Test images:",
    len(test_records)
)


# 13. Check Class Distribution by Split


In [ ]:
def class_counts(
    records,
    num_classes=3
):
    counts = [
        0
        for _ in range(
            num_classes
        )
    ]

    for record in records:
        counts[
            record["label"]
        ] += 1

    return counts

print(
    "Train:",
    class_counts(
        train_records
    )
)

print(
    "Validation:",
    class_counts(
        val_records
    )
)

print(
    "Test:",
    class_counts(
        test_records
    )
)


# 14. Reusable Dataset Pipeline

A reusable `Dataset` should separate:

- Stored records
- Transform logic
- Label extraction

The dataset should return one:

$$
(image,\ target)
$$

pair.


In [ ]:
class RecordImageDataset(Dataset):
    def __init__(
        self,
        records,
        transform=None
    ):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(
            self.records
        )

    def __getitem__(
        self,
        index
    ):
        record = self.records[
            index
        ]

        image = record[
            "image"
        ].clone()

        label = torch.tensor(
            record[
                "label"
            ],
            dtype=torch.long
        )

        if self.transform is not None:
            image = self.transform(
                image
            )

        return (
            image,
            label
        )


# 15. Training-Derived Normalization

Normalization statistics should come from training data only.

We calculate:

$$
\mu_{train}
$$

and:

$$
\sigma_{train}
$$

from training images.


In [ ]:
train_stack = torch.stack([
    record["image"]
    for record in train_records
])

train_mean = train_stack.mean(
    dim=(0, 2, 3)
)

train_std = train_stack.std(
    dim=(0, 2, 3)
)

print(
    "Training mean:",
    train_mean
)

print(
    "Training std:",
    train_std
)


# 16. Simple Transform Functions

We will use:

- Random horizontal shift for training
- Training-derived normalization
- Deterministic normalization for validation/test

For real ultrasound, augmentation should be clinically plausible.


In [ ]:
class NormalizeTensor:
    def __init__(
        self,
        mean,
        std
    ):
        self.mean = mean.view(
            -1,
            1,
            1
        )

        self.std = std.view(
            -1,
            1,
            1
        )

    def __call__(
        self,
        image
    ):
        return (
            image
            - self.mean
        ) / (
            self.std
            + 1e-8
        )


class TrainTransform:
    def __init__(
        self,
        mean,
        std
    ):
        self.normalize = (
            NormalizeTensor(
                mean,
                std
            )
        )

    def __call__(
        self,
        image
    ):
        shift = int(
            torch.randint(
                -3,
                4,
                (1,)
            ).item()
        )

        image = torch.roll(
            image,
            shifts=shift,
            dims=2
        )

        return self.normalize(
            image
        )


eval_transform = NormalizeTensor(
    train_mean,
    train_std
)

train_transform = TrainTransform(
    train_mean,
    train_std
)


# 17. Create Datasets


In [ ]:
train_dataset = (
    RecordImageDataset(
        train_records,
        transform=train_transform
    )
)

val_dataset = (
    RecordImageDataset(
        val_records,
        transform=eval_transform
    )
)

test_dataset = (
    RecordImageDataset(
        test_records,
        transform=eval_transform
    )
)

print(
    len(train_dataset),
    len(val_dataset),
    len(test_dataset)
)


# 18. Create DataLoaders

Training loader:

- Shuffle = `True`

Validation/test:

- Shuffle = `False`


In [ ]:
generator = (
    torch.Generator()
    .manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

print(
    "Train batches:",
    len(train_loader)
)


# 19. Inspect One Batch Before Training

Never skip this step.


In [ ]:
batch_images, batch_targets = next(
    iter(train_loader)
)

print(
    "Images:",
    batch_images.shape
)

print(
    "Targets:",
    batch_targets.shape
)

print(
    "Image dtype:",
    batch_images.dtype
)

print(
    "Target dtype:",
    batch_targets.dtype
)


# 20. Baseline Model

We begin with a small CNN.

A baseline should be:

- Simple
- Correct
- Reproducible
- Easy to debug

Do not start with the most complicated architecture.


In [ ]:
class BaselineCNN(nn.Module):
    def __init__(
        self,
        num_classes=3
    ):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                1,
                16,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),
            nn.ReLU()
        )

        self.pool = (
            nn.AdaptiveAvgPool2d(
                1
            )
        )

        self.classifier = nn.Linear(
            64,
            num_classes
        )

    def forward(
        self,
        x
    ):
        x = self.features(
            x
        )

        x = self.pool(
            x
        )

        x = torch.flatten(
            x,
            start_dim=1
        )

        return self.classifier(
            x
        )

baseline_model = BaselineCNN(
    num_classes=3
)

print(
    baseline_model
)


# 21. Baseline Shape Check


In [ ]:
dummy = torch.randn(
    8,
    1,
    64,
    64
)

dummy_logits = baseline_model(
    dummy
)

print(
    "Output shape:",
    dummy_logits.shape
)


# 22. Parameter Count


In [ ]:
total_parameters = sum(
    p.numel()
    for p in baseline_model.parameters()
)

print(
    "Parameters:",
    total_parameters
)


# 23. Loss and Optimizer

This is a 3-class problem.

Use:

`CrossEntropyLoss`

with raw logits.

We will use AdamW.


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    baseline_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    criterion
)

print(
    optimizer
)


# 24. Device Setup


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

baseline_model = baseline_model.to(
    device
)

print(
    "Device:",
    device
)


# 25. Accuracy Helper


In [ ]:
def batch_correct_count(
    logits,
    targets
):
    predictions = logits.argmax(
        dim=1
    )

    return (
        predictions
        == targets
    ).sum().item()


# 26. Training Function


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, targets in loader:
        images = images.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()
        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_correct += (
            batch_correct_count(
                logits,
                targets
            )
        )

        total_samples += (
            batch_size
        )

    return (
        total_loss
        / total_samples,
        total_correct
        / total_samples
    )


# 27. Validation Function


In [ ]:
def evaluate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(
                device
            )

            targets = targets.to(
                device
            )

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = (
                targets.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            total_correct += (
                batch_correct_count(
                    logits,
                    targets
                )
            )

            total_samples += (
                batch_size
            )

    return (
        total_loss
        / total_samples,
        total_correct
        / total_samples
    )


# 28. Best-Checkpoint Training Loop


In [ ]:
def fit_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs
):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_loss = float(
        "inf"
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    best_epoch = 0

    for epoch in range(
        epochs
    ):
        train_loss, train_acc = (
            train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                device
            )
        )

        val_loss, val_acc = (
            evaluate_one_epoch(
                model,
                val_loader,
                criterion,
                device
            )
        )

        history[
            "train_loss"
        ].append(
            train_loss
        )

        history[
            "train_acc"
        ].append(
            train_acc
        )

        history[
            "val_loss"
        ].append(
            val_loss
        )

        history[
            "val_acc"
        ].append(
            val_acc
        )

        if val_loss < best_val_loss:
            best_val_loss = (
                val_loss
            )

            best_epoch = (
                epoch + 1
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc:.3f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Acc {val_acc:.3f}"
        )

    model.load_state_dict(
        best_state
    )

    return (
        history,
        best_val_loss,
        best_epoch
    )


# 29. Train the Baseline


In [ ]:
torch.manual_seed(42)

baseline_model = BaselineCNN(
    num_classes=3
).to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    baseline_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

baseline_history, baseline_best_loss, baseline_best_epoch = (
    fit_model(
        baseline_model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        device,
        epochs=12
    )
)

print(
    "Best epoch:",
    baseline_best_epoch
)


# 30. Plot Training Curves


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    baseline_history[
        "train_loss"
    ],
    label="Train loss"
)

plt.plot(
    baseline_history[
        "val_loss"
    ],
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline Training Curves")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    baseline_history[
        "train_acc"
    ],
    label="Train accuracy"
)

plt.plot(
    baseline_history[
        "val_acc"
    ],
    label="Validation accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline Accuracy Curves")
plt.legend()
plt.show()


# 31. Save the Baseline Checkpoint


In [ ]:
baseline_checkpoint_path = Path(
    "baseline_best_model.pth"
)

torch.save(
    baseline_model.state_dict(),
    baseline_checkpoint_path
)

print(
    "Saved:",
    baseline_checkpoint_path
)


# 32. Collect Validation Predictions

For metric analysis, save:

- Targets
- Logits
- Probabilities
- Predicted classes


In [ ]:
def collect_outputs(
    model,
    loader,
    device
):
    model.eval()

    all_targets = []
    all_logits = []

    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(
                device
            )

            logits = model(
                images
            )

            all_targets.append(
                targets.cpu()
            )

            all_logits.append(
                logits.cpu()
            )

    targets = torch.cat(
        all_targets
    )

    logits = torch.cat(
        all_logits
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    predictions = logits.argmax(
        dim=1
    )

    return (
        targets,
        logits,
        probabilities,
        predictions
    )

val_targets, val_logits, val_probabilities, val_predictions = (
    collect_outputs(
        baseline_model,
        val_loader,
        device
    )
)

print(
    val_targets.shape,
    val_logits.shape
)


# 33. Confusion Matrix


In [ ]:
def confusion_matrix_multiclass(
    targets,
    predictions,
    num_classes
):
    matrix = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.long
    )

    for target, prediction in zip(
        targets,
        predictions
    ):
        matrix[
            target.long(),
            prediction.long()
        ] += 1

    return matrix

val_confusion = confusion_matrix_multiclass(
    val_targets,
    val_predictions,
    num_classes=3
)

print(
    val_confusion
)


# 34. Per-Class Metrics


In [ ]:
def per_class_recall_f1(
    confusion
):
    results = []

    for class_index in range(
        confusion.shape[0]
    ):
        tp = confusion[
            class_index,
            class_index
        ].item()

        fn = (
            confusion[
                class_index,
                :
            ].sum().item()
            - tp
        )

        fp = (
            confusion[
                :,
                class_index
            ].sum().item()
            - tp
        )

        recall = (
            tp
            / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        precision = (
            tp
            / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        f1 = (
            2
            * precision
            * recall
            / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        results.append({
            "class":
                class_index,

            "precision":
                precision,

            "recall":
                recall,

            "f1":
                f1
        })

    return results

val_class_metrics = (
    per_class_recall_f1(
        val_confusion
    )
)

for item in val_class_metrics:
    print(
        item
    )


# 35. Macro F1


In [ ]:
macro_f1 = sum(
    item["f1"]
    for item in val_class_metrics
) / len(
    val_class_metrics
)

print(
    "Validation macro F1:",
    macro_f1
)


# 36. Why Use More Than Accuracy?

Accuracy answers:

> How often was the prediction correct?

Macro F1 asks:

> How balanced is performance across classes?

In imbalanced or clinically asymmetric tasks, accuracy alone can hide important failure modes.


# 37. Hyperparameter Experiments

A strong experiment changes one factor at a time.

We will compare:

- Baseline
- Smaller learning rate
- Larger weight decay

The goal is not to exhaustively optimize.

The goal is to demonstrate controlled experimentation.


In [ ]:
experiment_configs = [
    {
        "name":
            "baseline",

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-4
    },
    {
        "name":
            "lower_lr",

        "learning_rate":
            3e-4,

        "weight_decay":
            1e-4
    },
    {
        "name":
            "more_decay",

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-3
    }
]

print(
    experiment_configs
)


# 38. Experiment Runner


In [ ]:
def run_experiment(
    config
):
    torch.manual_seed(
        42
    )

    model = BaselineCNN(
        num_classes=3
    ).to(
        device
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config[
            "learning_rate"
        ],
        weight_decay=config[
            "weight_decay"
        ]
    )

    history, best_loss, best_epoch = (
        fit_model(
            model,
            train_loader,
            val_loader,
            criterion,
            optimizer,
            device,
            epochs=8
        )
    )

    targets, logits, probabilities, predictions = (
        collect_outputs(
            model,
            val_loader,
            device
        )
    )

    confusion = confusion_matrix_multiclass(
        targets,
        predictions,
        num_classes=3
    )

    class_metrics = per_class_recall_f1(
        confusion
    )

    macro_f1 = sum(
        item["f1"]
        for item in class_metrics
    ) / len(
        class_metrics
    )

    accuracy = (
        predictions
        == targets
    ).float().mean().item()

    return {
        "name":
            config["name"],

        "model":
            model,

        "history":
            history,

        "best_val_loss":
            best_loss,

        "best_epoch":
            best_epoch,

        "val_accuracy":
            accuracy,

        "val_macro_f1":
            macro_f1
    }


# 39. Run Controlled Experiments

This may take a short time.


In [ ]:
experiment_results = []

for config in experiment_configs:
    print()
    print(
        "=" * 60
    )

    print(
        "Experiment:",
        config["name"]
    )

    result = run_experiment(
        config
    )

    experiment_results.append(
        result
    )


# 40. Compare Experiments


In [ ]:
for result in experiment_results:
    print(
        f"{result['name']:12s} | "
        f"Val Loss {result['best_val_loss']:.4f} | "
        f"Val Acc {result['val_accuracy']:.3f} | "
        f"Macro F1 {result['val_macro_f1']:.3f}"
    )


# 41. Model Selection

The model-selection rule should be defined before final testing.

For this notebook:

> Select the experiment with the lowest validation loss.

Do **not** use the test set to pick the winner.


In [ ]:
selected_result = min(
    experiment_results,
    key=lambda item:
        item["best_val_loss"]
)

selected_model = (
    selected_result[
        "model"
    ]
)

print(
    "Selected experiment:",
    selected_result[
        "name"
    ]
)


# 42. Transfer-Learning Baseline — Concept

A real project should often compare:

- Model trained from scratch
- Pretrained model / transfer-learning baseline

The question is:

> Does pretraining improve validation performance under the same split and evaluation protocol?


# 43. Why We Use a Lightweight Transfer-Learning Proxy Here

This notebook should run without internet access.

So instead of requiring a downloaded pretrained model, we will demonstrate the *experimental structure* using a frozen feature extractor.

In a real project, replace this with:

- ResNet
- DenseNet
- EfficientNet
- Another pretrained backbone

as studied in Notebook 19.


In [ ]:
class FrozenFeatureBaseline(nn.Module):
    def __init__(
        self,
        num_classes=3
    ):
        super().__init__()

        self.backbone = nn.Sequential(
            nn.Conv2d(
                1,
                16,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def forward(
        self,
        x
    ):
        x = self.backbone(
            x
        )

        x = torch.flatten(
            x,
            start_dim=1
        )

        return self.classifier(
            x
        )

transfer_proxy = (
    FrozenFeatureBaseline(
        num_classes=3
    )
)


# 44. Simulate a Frozen Backbone

In true transfer learning, the backbone would contain pretrained weights.

Here we freeze the backbone only to show the mechanics.


In [ ]:
for parameter in (
    transfer_proxy
    .backbone
    .parameters()
):
    parameter.requires_grad = False

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in transfer_proxy.parameters()
        if p.requires_grad
    )
)


# 45. Train Only the New Head


In [ ]:
transfer_proxy = transfer_proxy.to(
    device
)

transfer_optimizer = (
    torch.optim.Adam(
        filter(
            lambda p:
                p.requires_grad,
            transfer_proxy.parameters()
        ),
        lr=1e-3
    )
)

transfer_history, transfer_best_loss, transfer_best_epoch = (
    fit_model(
        transfer_proxy,
        train_loader,
        val_loader,
        criterion,
        transfer_optimizer,
        device,
        epochs=8
    )
)

print(
    "Transfer-style baseline best val loss:",
    transfer_best_loss
)


# 46. Fair Scratch vs Transfer Comparison

For a fair real comparison, keep fixed:

- Patient split
- Input size
- Label mapping
- Validation metric
- Checkpoint rule
- Evaluation code

Then change:

> Initialization / pretraining strategy


# 47. Error Analysis

After model selection, do not stop at a single metric.

Inspect:

- False predictions
- High-confidence errors
- Class-specific errors
- Patient-specific clusters
- Site/device patterns


In [ ]:
val_targets, val_logits, val_probabilities, val_predictions = (
    collect_outputs(
        selected_model,
        val_loader,
        device
    )
)

wrong_mask = (
    val_predictions
    != val_targets
)

wrong_indices = torch.where(
    wrong_mask
)[0]

print(
    "Validation errors:",
    len(
        wrong_indices
    )
)


# 48. High-Confidence Errors


In [ ]:
confidence = val_probabilities.max(
    dim=1
).values

high_conf_wrong = (
    wrong_mask
    & (
        confidence
        >= 0.8
    )
)

high_conf_wrong_indices = torch.where(
    high_conf_wrong
)[0]

print(
    "High-confidence errors:",
    len(
        high_conf_wrong_indices
    )
)


# 49. Map Validation Predictions Back to Records

Because validation loader is not shuffled, output order matches `val_records`.


In [ ]:
error_records = []

for index in wrong_indices.tolist():
    record = val_records[
        index
    ]

    error_records.append({
        "patient_id":
            record["patient_id"],

        "true_label":
            int(
                val_targets[
                    index
                ].item()
            ),

        "predicted_label":
            int(
                val_predictions[
                    index
                ].item()
            ),

        "confidence":
            float(
                confidence[
                    index
                ].item()
            )
    })

print(
    error_records[
        :5
    ]
)


# 50. Visualize a Few Errors


In [ ]:
num_to_show = min(
    6,
    len(
        wrong_indices
    )
)

if num_to_show > 0:
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(9, 6)
    )

    for axis_index, axis in enumerate(
        axes.flat
    ):
        if axis_index < num_to_show:
            index = int(
                wrong_indices[
                    axis_index
                ].item()
            )

            image = val_records[
                index
            ][
                "image"
            ]

            axis.imshow(
                image.squeeze(0).numpy(),
                cmap="gray"
            )

            axis.set_title(
                f"T={val_targets[index].item()} "
                f"P={val_predictions[index].item()} "
                f"C={confidence[index].item():.2f}"
            )

        axis.axis(
            "off"
        )

    plt.tight_layout()
    plt.show()

else:
    print(
        "No validation errors to display."
    )


# 51. Explainability Check

For a real vision project, inspect whether the selected model appears to use:

- Intended anatomy
- Suspicious corners
- Borders
- Text
- Device markers

We will use a lightweight input-gradient saliency check.


In [ ]:
def input_saliency(
    model,
    image,
    class_index=None
):
    model.eval()

    x = (
        image
        .detach()
        .clone()
        .to(device)
    )

    x.requires_grad_(
        True
    )

    model.zero_grad(
        set_to_none=True
    )

    logits = model(
        x
    )

    if class_index is None:
        class_index = (
            logits.argmax(
                dim=1
            ).item()
        )

    score = logits[
        0,
        class_index
    ]

    score.backward()

    saliency = (
        x.grad
        .detach()
        .abs()
        .cpu()
    )

    saliency = (
        saliency
        / (
            saliency.max()
            + 1e-8
        )
    )

    return (
        saliency,
        class_index
    )


# 52. Saliency for One Validation Sample


In [ ]:
example_record = val_records[
    0
]

example_image = eval_transform(
    example_record[
        "image"
    ]
).unsqueeze(
    0
)

saliency, explained_class = (
    input_saliency(
        selected_model,
        example_image
    )
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(7, 3)
)

axes[0].imshow(
    example_record[
        "image"
    ].squeeze(0).numpy(),
    cmap="gray"
)

axes[0].set_title(
    "Original image"
)

axes[1].imshow(
    saliency[
        0,
        0
    ].numpy(),
    cmap="hot"
)

axes[1].set_title(
    f"Saliency — Class {explained_class}"
)

for axis in axes:
    axis.axis(
        "off"
    )

plt.tight_layout()
plt.show()


# 53. Explainability Is a Debugging Tool

Do not conclude:

> The model has proven clinically correct reasoning.

Instead ask:

- Is the highlighted region plausible?
- Is the model focusing on borders?
- Is there a suspicious shortcut?
- Does masking the region change the prediction?

Use explanations to generate testable hypotheses.


# 54. Reproducibility

A reliable project records:

- Random seed
- Split IDs
- Hyperparameters
- Model name
- Preprocessing
- Checkpoint metric
- Software version


In [ ]:
experiment_metadata = {
    "project":
        "synthetic_ultrasound_style_classification",

    "seed":
        42,

    "num_classes":
        3,

    "image_size":
        64,

    "split_unit":
        "patient",

    "checkpoint_metric":
        "validation_loss",

    "selected_experiment":
        selected_result[
            "name"
        ],

    "torch_version":
        torch.__version__
}

print(
    json.dumps(
        experiment_metadata,
        indent=2
    )
)


# 55. Save Split Definitions

Saving only the random seed may be insufficient if the dataset later changes.

Save exact IDs.


In [ ]:
split_manifest = {
    "train_patient_ids":
        sorted(
            train_patient_ids
        ),

    "val_patient_ids":
        sorted(
            val_patient_ids
        ),

    "test_patient_ids":
        sorted(
            test_patient_ids
        )
}

Path(
    "split_manifest.json"
).write_text(
    json.dumps(
        split_manifest,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Split manifest saved."
)


# 56. Save Experiment Configuration


In [ ]:
Path(
    "experiment_metadata.json"
).write_text(
    json.dumps(
        experiment_metadata,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Experiment metadata saved."
)


# 57. Save Validation Predictions

Saving predictions makes metric analysis reproducible without rerunning inference.


In [ ]:
validation_prediction_records = []

for index, record in enumerate(
    val_records
):
    validation_prediction_records.append({
        "patient_id":
            int(
                record[
                    "patient_id"
                ]
            ),

        "image_index":
            int(
                record[
                    "image_index"
                ]
            ),

        "true_label":
            int(
                val_targets[
                    index
                ].item()
            ),

        "predicted_label":
            int(
                val_predictions[
                    index
                ].item()
            ),

        "probabilities":
            [
                float(value)
                for value in val_probabilities[
                    index
                ].tolist()
            ]
    })

Path(
    "validation_predictions.json"
).write_text(
    json.dumps(
        validation_prediction_records,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Validation predictions saved."
)


# 58. Final Test Evaluation

Only now, after:

- Model selection
- Hyperparameter choices
- Checkpoint selection

do we evaluate the test set.

The test set should not guide further tuning.


In [ ]:
test_loss, test_accuracy = (
    evaluate_one_epoch(
        selected_model,
        test_loader,
        criterion,
        device
    )
)

test_targets, test_logits, test_probabilities, test_predictions = (
    collect_outputs(
        selected_model,
        test_loader,
        device
    )
)

test_confusion = confusion_matrix_multiclass(
    test_targets,
    test_predictions,
    num_classes=3
)

test_class_metrics = (
    per_class_recall_f1(
        test_confusion
    )
)

test_macro_f1 = sum(
    item["f1"]
    for item in test_class_metrics
) / len(
    test_class_metrics
)

print(
    "Test loss:",
    test_loss
)

print(
    "Test accuracy:",
    test_accuracy
)

print(
    "Test macro F1:",
    test_macro_f1
)

print(
    "Test confusion matrix:"
)

print(
    test_confusion
)


# 59. Save Final Test Predictions


In [ ]:
test_prediction_records = []

for index, record in enumerate(
    test_records
):
    test_prediction_records.append({
        "patient_id":
            int(
                record[
                    "patient_id"
                ]
            ),

        "image_index":
            int(
                record[
                    "image_index"
                ]
            ),

        "true_label":
            int(
                test_targets[
                    index
                ].item()
            ),

        "predicted_label":
            int(
                test_predictions[
                    index
                ].item()
            ),

        "probabilities":
            [
                float(value)
                for value in test_probabilities[
                    index
                ].tolist()
            ]
    })

Path(
    "test_predictions.json"
).write_text(
    json.dumps(
        test_prediction_records,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Test predictions saved."
)


# 60. Final Model Checkpoint


In [ ]:
final_checkpoint = {
    "model_state_dict":
        selected_model.state_dict(),

    "metadata":
        experiment_metadata,

    "test_metrics":
        {
            "loss":
                float(
                    test_loss
                ),

            "accuracy":
                float(
                    test_accuracy
                ),

            "macro_f1":
                float(
                    test_macro_f1
                )
        }
}

torch.save(
    final_checkpoint,
    "final_project_checkpoint.pth"
)

print(
    "Final checkpoint saved."
)


# 61. What Should Be Saved in a Real Project?

At minimum:

- Best model weights
- Experiment configuration
- Exact train/validation/test split
- Class mapping
- Preprocessing definition
- Final predictions
- Metrics
- Random seed
- Software versions


# 62. Complete Project Organization

A useful project structure:

```text
project/
│
├── data/
│   ├── raw/
│   └── processed/
│
├── splits/
│   ├── train.csv
│   ├── val.csv
│   └── test.csv
│
├── notebooks/
│
├── src/
│   ├── datasets.py
│   ├── transforms.py
│   ├── models.py
│   ├── train.py
│   ├── evaluate.py
│   ├── explain.py
│   └── utils.py
│
├── configs/
│
├── checkpoints/
│
├── predictions/
│
├── reports/
│
├── requirements.txt
└── README.md
```


# 63. `datasets.py`

This file can contain:

- Dataset classes
- File loading
- Label mapping
- Patient IDs
- Metadata parsing


# 64. `transforms.py`

This file can contain:

- Resize
- Normalization
- Augmentation
- Grayscale conversion
- Cropping
- Device/site harmonization steps

Keep preprocessing explicit and reproducible.


# 65. `models.py`

This file can contain:

- Baseline CNN
- Transfer-learning model
- Model factory
- Weight initialization
- Head replacement


# 66. `train.py`

This file can contain:

- `train_one_epoch`
- `evaluate_one_epoch`
- Checkpointing
- Early stopping
- Scheduler logic
- Logging


# 67. `evaluate.py`

This file can contain:

- Prediction collection
- Confusion matrix
- Accuracy
- Macro F1
- AUROC / AUPRC
- Calibration
- Subgroup metrics


# 68. `explain.py`

This file can contain:

- Saliency
- Grad-CAM
- Occlusion sensitivity
- Failure-case visualization


# 69. Preparing for Real Ultrasound Data

Now replace the synthetic records with real metadata.

A useful metadata table might contain:

$$
\begin{array}{|c|c|}
\hline
path & \text{Image file path} \\
\hline
label & \text{Target class} \\
\hline
patient\_id & \text{Patient identifier} \\
\hline
study\_id & \text{Study/exam identifier} \\
\hline
site & \text{Hospital/site} \\
\hline
device & \text{Scanner/device} \\
\hline
view & \text{Ultrasound view} \\
\hline
\end{array}
$$


# 70. Real Ultrasound Split Design

Before training, decide whether splitting should be by:

- Patient
- Study
- Site
- Device
- Time period

For generalization research, consider:

- Internal patient-level validation
- External-site testing
- Device-held-out testing

depending on the scientific objective.


# 71. Real Ultrasound Preprocessing

Document every step.

Possible steps:

- Load file
- Convert grayscale
- Remove borders
- Remove text overlays
- Crop relevant field of view
- Resize
- Normalize
- Augment training data

Every preprocessing decision can influence the learned model.


# 72. Ultrasound Harmonization

If images come from multiple scanners or sites, differences may include:

- Intensity
- Contrast
- Noise
- Resolution
- Gain
- Dynamic range
- Post-processing

Harmonization should reduce irrelevant acquisition variation without erasing clinically relevant signal.

Always evaluate whether harmonization improves:

- Cross-site generalization
- Device robustness
- Calibration
- Class performance

rather than only visual appearance.


# 73. Harmonization Must Be Fit Without Leakage

If a harmonization method estimates parameters from data:

> Fit those parameters using training data only.

Then apply the learned transformation to validation and test data.

Otherwise preprocessing can leak evaluation information.


# 74. Multi-Site Ultrasound Evaluation

For each site, report:

- Number of patients
- Class distribution
- Accuracy
- Macro F1
- Sensitivity/specificity where relevant
- AUROC/AUPRC where relevant

Overall performance can hide site-specific failure.


# 75. Device-Level Error Analysis

If one device performs poorly:

Investigate:

- Resolution
- Intensity scale
- Probe type
- Cropping
- Overlays
- Acquisition protocol
- Harmonization

Then compare explanation maps and errors by device.


# 76. Patient-Level Prediction

If your model predicts per image but the clinical target is per patient, define an aggregation rule.

Possible examples:

- Mean probability
- Maximum probability
- Majority vote
- Learned study-level aggregation

This rule should be defined and validated before final testing.


# 77. Example Patient-Level Probability Aggregation


In [ ]:
def aggregate_patient_probabilities(
    prediction_records,
    num_classes=3
):
    patient_to_probs = {}

    for record in prediction_records:
        patient_id = record[
            "patient_id"
        ]

        probability = torch.tensor(
            record[
                "probabilities"
            ]
        )

        patient_to_probs.setdefault(
            patient_id,
            []
        ).append(
            probability
        )

    aggregated = {}

    for patient_id, values in (
        patient_to_probs.items()
    ):
        mean_probability = torch.stack(
            values
        ).mean(
            dim=0
        )

        aggregated[
            patient_id
        ] = {
            "probabilities":
                mean_probability,

            "prediction":
                int(
                    mean_probability.argmax(
                        dim=0
                    ).item()
                )
        }

    return aggregated

patient_predictions = (
    aggregate_patient_probabilities(
        test_prediction_records
    )
)

print(
    list(
        patient_predictions.items()
    )[:3]
)


# 78. Do Not Mix Image-Level and Patient-Level Metrics

If the deployment decision is per patient:

> Report patient-level metrics.

You can also report image-level metrics, but label them clearly.

The unit of analysis must match the interpretation of the result.


# 79. Reproducibility Checklist

Before considering the project complete, verify:

1. Fixed random seed
2. Saved split manifest
3. Saved config
4. Saved model checkpoint
5. Saved predictions
6. Saved metric definitions
7. Recorded preprocessing
8. Recorded software versions
9. Re-runnable training script
10. Re-runnable evaluation script


# 80. Reliability Checklist

A model is not reliable just because validation accuracy is high.

Also verify:

- No leakage
- No patient overlap
- No test tuning
- Class-wise performance
- Error analysis
- Site/device robustness
- Calibration if probabilities matter
- Explainability sanity checks


# 81. Common Mistake — Starting With a Huge Model

A large model makes debugging harder.

Start with a simple baseline.

Then increase complexity only when the baseline is understood.


# 82. Common Mistake — Random Image-Level Splits in Patient Data

This can leak patient-specific information.

Split by the independent unit.


# 83. Common Mistake — Hyperparameter Tuning on the Test Set

The test set is for final evaluation.

Use validation data for development decisions.


# 84. Common Mistake — Comparing Models on Different Splits

If model A and model B use different patient splits, the comparison is less controlled.

Use the same split for fair comparisons.


# 85. Common Mistake — Saving Only Accuracy

Save predictions.

Predictions let you recompute:

- Confusion matrix
- F1
- AUROC
- Calibration
- Subgroup metrics


# 86. Common Mistake — Ignoring Class Mapping

Always save a mapping such as:

```python
{
    0: "Class A",
    1: "Class B",
    2: "Class C"
}
```

A checkpoint without label meaning is incomplete.


# 87. Common Mistake — Changing Preprocessing After Model Selection

If you change normalization, crop, or resize, you changed the experiment.

Re-evaluate the model under the new pipeline.


# 88. Common Mistake — Treating Explainability as Final Validation

Heatmaps are not a substitute for:

- External testing
- Robustness evaluation
- Calibration
- Error analysis


# 89. Common Mistake — Ignoring External Generalization

A model can perform well internally and poorly on a new site.

For ultrasound, external validation is especially valuable because scanner and acquisition differences can be large.


# 90. Common Mistake — Not Keeping Raw Data Immutable

Keep raw files unchanged.

Generate processed data separately.

This protects reproducibility.


# 91. A Compact End-to-End Workflow

The complete workflow is:

$$
\boxed{
\begin{array}{c}
Define\ Task\\
\downarrow\\
Define\ Split\ Unit\\
\downarrow\\
Create\ Patient-Level\ Splits\\
\downarrow\\
Fit\ Training-Only\ Preprocessing\\
\downarrow\\
Build\ Dataset/DataLoader\\
\downarrow\\
Train\ Baseline\\
\downarrow\\
Validate\\
\downarrow\\
Run\ Controlled\ Experiments\\
\downarrow\\
Select\ Best\ Model\\
\downarrow\\
Error/Explainability\ Analysis\\
\downarrow\\
Freeze\ Decisions\\
\downarrow\\
Final\ Test
\end{array}
}
$$


# 92. Practice Exercises

## Exercise 1

Define a binary ultrasound classification task including:

- Input
- Target
- Prediction unit
- Primary metric

## Exercise 2

Create patient-level train/validation/test splits.

## Exercise 3

Write assertions proving the patient sets are disjoint.

## Exercise 4

Create a custom image `Dataset`.

## Exercise 5

Compute normalization statistics from training data only.

## Exercise 6

Build a small baseline CNN.

## Exercise 7

Write reusable training and validation functions.

## Exercise 8

Save the best validation checkpoint.

## Exercise 9

Save predictions and experiment configuration.

## Exercise 10

Write a final test-evaluation pipeline that does not tune anything.


# 93. Conceptual Challenges

## Challenge 1

Why should task definition come before architecture choice?

## Challenge 2

Why is patient-level splitting essential for many medical-imaging datasets?

## Challenge 3

Why should preprocessing statistics be fit only on training data?

## Challenge 4

Why should a simple baseline be trained before a complex model?

## Challenge 5

Why should validation loss and test loss have different roles?

## Challenge 6

Why must hyperparameter experiments use the same split?

## Challenge 7

Why should predictions be saved, not only summary metrics?

## Challenge 8

Why can explainability help debug preprocessing?

## Challenge 9

Why does harmonization require leakage-aware design?

## Challenge 10

Why can image-level metrics differ from patient-level metrics?

## Challenge 11

Why should external-site performance be evaluated for ultrasound?

## Challenge 12

Why is the final test set used only after all development decisions are frozen?


# 94. Exercise Solutions


In [ ]:
# Exercise 2 and 3
patient_ids = list(
    range(30)
)

random.Random(
    7
).shuffle(
    patient_ids
)

train_ids = set(
    patient_ids[:20]
)

val_ids = set(
    patient_ids[20:25]
)

test_ids = set(
    patient_ids[25:]
)

assert train_ids.isdisjoint(
    val_ids
)

assert train_ids.isdisjoint(
    test_ids
)

assert val_ids.isdisjoint(
    test_ids
)

print(
    "Patient-level split verified."
)


In [ ]:
# Exercise 4
class SimpleImageDataset(Dataset):
    def __init__(
        self,
        images,
        targets
    ):
        self.images = images
        self.targets = targets

    def __len__(self):
        return len(
            self.images
        )

    def __getitem__(
        self,
        index
    ):
        return (
            self.images[
                index
            ],
            self.targets[
                index
            ]
        )


In [ ]:
# Exercise 5
exercise_train_images = torch.randn(
    20,
    1,
    32,
    32
)

exercise_mean = (
    exercise_train_images.mean(
        dim=(0, 2, 3)
    )
)

exercise_std = (
    exercise_train_images.std(
        dim=(0, 2, 3)
    )
)

print(
    exercise_mean,
    exercise_std
)


In [ ]:
# Exercise 6
exercise_model = nn.Sequential(
    nn.Conv2d(
        1,
        8,
        3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(
        8,
        16,
        3,
        padding=1
    ),
    nn.ReLU(),

    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(
        16,
        2
    )
)

print(
    exercise_model
)


In [ ]:
# Exercise 9
exercise_config = {
    "seed": 42,
    "split_unit": "patient",
    "model": "small_cnn",
    "primary_metric": "validation_loss"
}

Path(
    "exercise_project_config.json"
).write_text(
    json.dumps(
        exercise_config,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Exercise configuration saved."
)


# 95. Key Takeaways

In this notebook, we connected the complete PyTorch workflow:

- Prediction-task definition
- Unit of analysis
- Primary metric definition
- Patient-level splitting
- Leakage prevention
- Training-derived preprocessing
- Custom datasets
- DataLoaders
- Baseline CNN
- Training
- Validation
- Checkpointing
- Controlled hyperparameter experiments
- Model selection
- Transfer-learning comparison structure
- Error analysis
- High-confidence-error inspection
- Explainability checks
- Reproducibility
- Split manifests
- Saving predictions
- Experiment metadata
- Final test evaluation
- Project organization
- Ultrasound pipeline preparation
- Harmonization considerations
- Patient-level aggregation
- External-site evaluation

The most important project principle is:

$$
\boxed{
\text{Reliable Model}
=
\text{Reliable Data Split}
+
\text{Reproducible Pipeline}
+
\text{Controlled Validation}
+
\text{Untouched Test Set}
}
$$

And for medical imaging:

$$
\boxed{
\text{No Patient Leakage}
}
$$

is a fundamental requirement.


# 96. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. Why define the prediction task before choosing a model?
2. What is the unit of analysis?
3. Why should the primary metric be defined before final testing?
4. Why split by patient instead of image in many ultrasound datasets?
5. Why should normalization statistics come only from training data?
6. Why should validation and test transforms usually be deterministic?
7. Why begin with a simple baseline?
8. What is the purpose of validation loss?
9. Why save the best checkpoint instead of only the last epoch?
10. Why should hyperparameter experiments use the same split?
11. Why should the test set remain untouched during model development?
12. Why save predictions?
13. Why save exact split IDs?
14. Why save experiment metadata?
15. Why compare scratch training and transfer learning?
16. Why inspect high-confidence errors?
17. Why use explainability as a debugging tool?
18. Why can harmonization leak information?
19. Why can site/device differences matter in ultrasound?
20. Why might patient-level aggregation be needed?
21. Why are image-level and patient-level metrics different?
22. Why should external-site evaluation be considered?
23. What belongs in `datasets.py`?
24. What belongs in `models.py`?
25. What belongs in `train.py`?
26. What belongs in `evaluate.py`?
27. What belongs in `explain.py`?
28. Why keep raw data immutable?
29. Why should preprocessing changes create a new experiment?
30. What makes a PyTorch project scientifically reliable?


# Next Notebook

# 24 — Building a Real Ultrasound Classification Pipeline

In the next notebook, we will study:

- Reading image metadata from CSV
- Patient-level split manifests
- File-path based custom datasets
- Loading grayscale ultrasound images
- Resizing and normalization
- Training-only augmentation
- Handling variable image sizes
- Class imbalance
- Weighted losses and sampling
- Baseline CNN for ultrasound
- Transfer-learning baseline
- Site/device-aware evaluation
- Harmonization experiments
- Patient-level prediction aggregation
- Saving reproducible experiment outputs
- Preparing for a real research-quality ultrasound study
